# 01 — Análise Exploratória de Dados (EDA)
**Tech Challenge Fase 4 · POSTECH DTAT · Grupo 12**

## Objetivo
Descrever a **origem como ela é**, antes de qualquer tratamento. Este notebook não corrige,
não deduplica e não decide nada: ele **mede** a qualidade do `amazon.csv` e produz a evidência
que justifica cada regra aplicada em `02_tratamento`. Por isso a leitura aqui é do **bronze**,
e não da camada analítica — ler o gold nesta etapa esconderia exatamente o que a EDA precisa expor.

## Entradas
| Caminho | Camada | O que é |
|---|---|---|
| `00_data/bronze/amazon.csv` | bronze | origem intocada · 1.465 × 16 · **tudo texto** · **somente leitura** |
| `src/paleta.py` | — | paleta v06 aplicada por ferramenta (T9) |

## Saídas
| Caminho | O que é |
|---|---|
| `02_results/images/fig_eda_qualidade_origem.png` | as verificações de integridade da origem |
| `02_results/images/fig_eda_distribuicoes.png` | distribuição das cinco variáveis numéricas |
| `02_results/images/fig_eda_outliers.png` | dispersão e pontos fora da cerca de Tukey (IQR) |

> **Este notebook nunca escreve em `00_data/`.** A origem é lida com `dtype=str` e permanece intacta.

## Roteiro
1. Ambiente e paleta · 2. Forma e tipos · 3. Nulos por coluna · 4. Duplicidade ·
5. Falhas de conversão · 6. Coerência de preço · 7. Desconto declarado × recalculado ·
8. Outliers por IQR · 9. Distribuições · 10. Figuras · 11. **Validação e reconciliação**

---
## ⚠️ As oito limitações obrigatórias (L1 a L8)
Exigência de `method/limitations/limitacoes_obrigatorias.md`: as oito abrem o notebook, porque
delimitam **o que este conjunto de dados pode responder**. Elas não são ressalva de rodapé — são
o contrato de leitura de todo número gerado daqui para a frente.

**L1 · Truncamento em 8 avaliações por produto.** A base exibe no máximo **8 avaliações por produto**
(média medida: **7,85**). O texto analisado **não é amostra aleatória** do que os consumidores
escreveram: é o recorte que o marketplace escolheu exibir na vitrine.
→ *Consequência:* os achados textuais descrevem **os temas visíveis na vitrine**, nunca
"o que a maioria dos clientes pensa".

**L2 · Idioma do corpus.** As avaliações são predominantemente em inglês, com ocorrências em outros
idiomas e alfabetos não latinos, que o pipeline remove.
→ *Consequência:* os temas representam o subconjunto em alfabeto latino; a parcela descartada não
foi quantificada por idioma.

**L3 · Concentração de categorias.** Das 9 categorias de nível 1, apenas **4** têm 30 produtos ou mais
— Electronics (490), Home & Kitchen (447), Computers & Accessories (373) e Office Products (31).
As **outras 5 somam 7 produtos**.
→ *Consequência:* "análise de portfólio por categoria" significa, na prática, **três famílias mais
uma no limite amostral**. Nenhuma recomendação se estende às categorias suprimidas.

**L4 · Ausência de variável de venda.** A base **não contém** unidades vendidas, receita, margem,
estoque ou data. `rating_count` é **volume de avaliações** — proxy de engajamento, nunca de vendas (T6).
→ *Consequência:* nenhuma recomendação em receita ou volume vendido, e **nenhuma análise temporal**
— não há afirmação possível sobre tendência, sazonalidade ou evolução.

**L5 · Mercado e moeda.** Marketplace **indiano**, valores em **rupia indiana**, **sem conversão**.
→ *Consequência:* as faixas de preço descrevem aquele mercado. Transposição para outro mercado é
analogia, não resultado. Neste projeto os valores são publicados **sem símbolo de moeda** — o símbolo
só existe dentro do bronze, que é texto de origem. No único nome de produto que o traz, o
tratamento o substitui pela palavra `rupia` (L5, exceção 2).

**L6 · Inconsistências da origem.** A base traz **114 linhas excedentes** em **92 `product_id`**, das
quais **31** têm `rating_count` divergente entre as ocorrências. A regra adotada mantém a **maior**
ocorrência, com desempate por menor índice de linha.
→ *Impacto medido:* diferença agregada de **40 avaliações em 23,8 milhões** — imaterial. Nota,
categoria e nome **nunca** divergem; o preço de tabela diverge em 1 produto e o preço com
desconto em 3.

**L7 · Amostragem e generalização.** Os dados descrevem os produtos deste recorte do catálogo, não o
catálogo completo nem o e-commerce em geral. **Não há pesos amostrais.**

**L8 · Listagens variantes compartilham avaliações.** **254 produtos** (18,8%) em **96 grupos** repetem
exatamente o mesmo par nota × volume: são listagens do mesmo item (cor, capacidade, embalagem). O
volume delas é o mesmo contador visto mais de uma vez.
→ *Consequência:* toda leitura de participação por categoria declara as **duas** versões, com e sem
variantes.

---
### Regras transversais que valem para tudo o que este notebook imprime
`T3` todo número declara base e *n* · `T4` **vedada linguagem causal** · `T5` magnitude sempre junto
com significância · `T6` `rating_count` é volume de avaliações · `T7` `user_id` e `user_name`
**nunca** aparecem em saída alguma · `T9` toda figura usa a paleta v06 aplicada por ferramenta.

---
## 1. Ambiente e paleta

A paleta é aplicada **antes do primeiro gráfico** e **por ferramenta**, nunca à mão: é exatamente o
que T9 exige. Sem `aplicar()`, o matplotlib usa o ciclo de cores padrão da biblioteca — o que já
configura descumprimento, mesmo que o gráfico "fique bonito".

A raiz do repositório é resolvida a partir do diretório de execução, para que o notebook funcione
tanto executado de dentro de `01_notebooks/` quanto da raiz.

In [1]:
# --- localização da raiz do repositório (robusta ao diretório de execução) ---
import os, sys
from pathlib import Path

RAIZ = Path.cwd()
if not (RAIZ / "00_data").exists():          # executado de dentro de 01_notebooks/
    RAIZ = RAIZ.parent
assert (RAIZ / "00_data" / "bronze" / "amazon.csv").exists(), f"bronze não encontrado a partir de {RAIZ}"

sys.path.insert(0, str(RAIZ / "src"))        # T9 — a paleta vem da ferramenta, não do gosto

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")                        # execução sem display (nbconvert)
import matplotlib.pyplot as plt

from paleta import aplicar, confere_visual, CATEGORICA, CHAMADAS, ESTRUTURA, BASE, checar_series
aplicar()                                    # <-- sem esta linha, T9 está descumprida

BRONZE = RAIZ / "00_data" / "bronze" / "amazon.csv"
IMG    = RAIZ / "02_results" / "images"
IMG.mkdir(parents=True, exist_ok=True)

# Apelidos da paleta v06 — nenhuma cor é escrita à mão neste notebook.
AMBAR, AZUL, CLARA = CATEGORICA[0], CATEGORICA[1], CATEGORICA[2]
LARANJA, TEAL      = CHAMADAS["atencao"], CHAMADAS["favoravel"]
TINTA, SECUNDARIO  = ESTRUTURA["tinta"], ESTRUTURA["secundario"]

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 30)

print(f"raiz      : {RAIZ.name}/")
print(f"bronze    : {BRONZE.relative_to(RAIZ)}")
print(f"pandas    : {pd.__version__} · numpy {np.__version__} · matplotlib {matplotlib.__version__}")
print("paleta v06 aplicada (T9)")

raiz      : Techchallenge4-G12/
bronze    : 00_data/bronze/amazon.csv
pandas    : 3.0.6 · numpy 2.5.3 · matplotlib 3.11.2
paleta v06 aplicada (T9)


### Funções de apoio

Duas decisões ficam explícitas aqui:

- **`num()`** reproduz literalmente a conversão do dicionário de dados — remove o símbolo de moeda,
  o separador de milhar e o `%`, e usa `errors="coerce"` para que **a falha de conversão apareça como
  nulo em vez de derrubar a execução**. É esse `coerce` que transforma erro em medida: é assim que o
  `rating` inválido e os `rating_count` nulos ficam contáveis.
- **`rodape()`** aplica T3 em toda figura: fonte, base e *n*. Nenhuma figura sai sem ele.

In [2]:
def num(serie, remover):
    """Converte texto em número removendo símbolos. Falha vira nulo (coerce), não exceção."""
    saida = serie.astype(str)
    for ch in remover:
        saida = saida.str.replace(ch, "", regex=False)
    return pd.to_numeric(saida, errors="coerce")


def br(x, casas=0):
    """Formata número no padrão brasileiro: 1.465 · 7,85."""
    s = f"{x:,.{casas}f}"
    return s.replace(",", "§").replace(".", ",").replace("§", ".")


def rodape(fig, texto, folga=None):
    """T3 — toda figura declara fonte, base e n. Abaixo do eixo, nunca sobre ele.

    `folga` reserva margem inferior antes de escrever: sem ela o rodapé colide
    com o rótulo do eixo x em figuras de painel único.
    """
    if folga is not None:
        fig.subplots_adjust(bottom=folga)
    fig.text(0.01, 0.005, texto, fontsize=7, color=SECUNDARIO, ha="left")


def salva(fig, nome):
    """Grava em 02_results/images/ com o fundo da paleta v06."""
    confere_visual(fig, nome)          # T3 e T9 conferidos na figura, não no código
    caminho = IMG / nome
    fig.savefig(caminho, dpi=150, bbox_inches="tight", pad_inches=0.28,
                facecolor=BASE["canvas"])
    plt.close(fig)
    print(f"  gravada: 02_results/images/{nome}")


FONTE = ("Fonte: Kaggle · karkavelrajaj/amazon-sales-dataset · leitura do bronze, sem tratamento")
MOEDA = "valores monetários em rupia indiana, sem conversão e sem símbolo (L5)"
print("funções de apoio definidas")

funções de apoio definidas


---
## 2. Forma e tipos

A leitura é feita com `dtype=str` **de propósito**. Deixar o pandas inferir tipos aqui seria começar
a tratar os dados antes de medi-los: a inferência silenciaria justamente as falhas que a EDA precisa
contar. Na origem, **as 16 colunas são texto** — preço com símbolo de moeda e separador de milhar,
desconto com `%`, avaliações com vírgula de milhar e o bloco de reviews concatenado.

**T7:** `user_id` e `user_name` existem no bronze, mas **não são impressos** em nenhuma saída deste
notebook. Elas aparecem apenas como *nome de coluna* na listagem de estrutura, nunca como conteúdo.

In [3]:
df = pd.read_csv(BRONZE, dtype=str)

print(f"forma: {df.shape[0]} linhas × {df.shape[1]} colunas")
print(f"esperado (base bruta): 1465 × 16 → {'OK' if df.shape == (1465, 16) else 'DIVERGENTE'}\n")

PESSOAIS = ["user_id", "user_name"]            # T7 — nunca exibidas
ESCRITO_PELO_CONSUMIDOR = ["review_title", "review_content"]   # T7 — nunca exibidas

# Três máscaras de EXIBIÇÃO, e nenhuma delas toca o dado:
#   T7/LGPD        -> user_id e user_name não são exibidos, nem como exemplo.
#   T7/ética       -> título e texto da avaliação também não. Um trecho de 44
#                      caracteres ainda é a frase de uma pessoa, e este notebook
#                      é versionado: o que a célula precisa mostrar é o FORMATO
#                      da coluna, não o que alguém escreveu. O corpus é lido
#                      sempre agregado, no notebook 05.
#   notação de L5   -> o marcador de moeda da origem é descrito, não desenhado.
#                      A premissa do projeto é que o símbolo da rupia não apareça
#                      em artefato nenhum; o bronze continua intacto (o SHA-256
#                      depende disso) e é só a exibição que troca o glifo pelo
#                      rótulo "(rupia)".
MOEDA_ORIGEM = "\u20b9"


def amostra(coluna):
    """Primeiro valor não nulo da coluna, pronto para exibir."""
    if coluna in PESSOAIS:
        return "[suprimido — T7/LGPD]"
    if coluna in ESCRITO_PELO_CONSUMIDOR:
        comp = df[coluna].dropna().str.len()
        return f"[texto escrito pelo consumidor — suprimido · {comp.min()} a {comp.max()} caracteres]"
    v = str(df[coluna].dropna().iloc[0])
    v = v.replace(MOEDA_ORIGEM, "(rupia) ")          # máscara de notação (L5)
    return v[:44] + "…" if len(v) > 44 else v


estrutura = pd.DataFrame({
    "coluna": df.columns,
    "dtype_origem": [str(t) for t in df.dtypes],
    "distintos": [df[c].nunique(dropna=True) for c in df.columns],
    "exemplo": [amostra(c) for c in df.columns],
})
estrutura

forma: 1465 linhas × 16 colunas
esperado (base bruta): 1465 × 16 → OK



,coluna,dtype_origem,distintos,exemplo
0,product_id,str,1351,B07JW9H4J1
1,product_name,str,1337,Wayona Nylon Braided USB to Lightning Fast C…
2,category,str,211,Computers&Accessories|Accessories&Peripheral…
3,discounted_price,str,550,(rupia) 399
4,actual_price,str,449,"(rupia) 1,099"
5,discount_percentage,str,92,64%
6,rating,str,28,4.2
7,rating_count,str,1143,"24,269"
8,about_product,str,1293,High Compatibility : Compatible With iPhone …
9,user_id,str,1194,[suprimido — T7/LGPD]


In [4]:
# Todas as colunas chegam como texto? (se alguma não for, a leitura não foi a esperada)
# `dtype=str` materializa como `object` no pandas 2 e como `str` no pandas 3: o que
# precisa ser verdade é que NENHUMA coluna chegou tipada como número ou data.
tipos_unicos = set(str(t) for t in df.dtypes)
texto = tipos_unicos <= {"object", "str", "string"}
print(f"pandas {pd.__version__} · tipos presentes na origem: {tipos_unicos}")
print(f"16 colunas todas texto → {'OK' if texto and df.shape[1] == 16 else 'DIVERGENTE'}")

pandas 3.0.6 · tipos presentes na origem: {'str'}
16 colunas todas texto → OK


---
## 3. Nulos por coluna

Nulo **declarado** (célula vazia na origem) é diferente de nulo **produzido pela conversão**
(texto que não vira número). Esta seção mede o primeiro; a seção 5 mede o segundo. Separar os dois
é o que permite dizer, com honestidade, de onde veio cada produto removido da base analítica.

In [5]:
nulos = pd.DataFrame({
    "nulos": df.isna().sum(),
    "% da base bruta": (df.isna().mean() * 100).round(3),
}).sort_values("nulos", ascending=False)

print(f"base: bruta · n = {br(len(df))}")
print(f"total de células nulas na origem: {int(df.isna().sum().sum())}\n")
nulos

base: bruta · n = 1.465
total de células nulas na origem: 2



,nulos,% da base bruta
rating_count,2,0.137
product_id,0,0.000
product_name,0,0.000
category,0,0.000
discounted_price,0,0.000
actual_price,0,0.000
discount_percentage,0,0.000
rating,0,0.000
about_product,0,0.000
user_id,0,0.000


In [6]:
col_com_nulo = nulos[nulos.nulos > 0]
if col_com_nulo.empty:
    print("nenhuma coluna com nulo declarado na origem")
else:
    for c, linha in col_com_nulo.iterrows():
        print(f"{c}: {int(linha.nulos)} nulo(s) — {linha['% da base bruta']:.3f}% da base bruta")
    print("\nLeitura: o único nulo declarado da origem está em rating_count —")
    print("volume de avaliações (T6), nunca vendas. Os 2 produtos afetados saem da base analítica.")

rating_count: 2 nulo(s) — 0.137% da base bruta

Leitura: o único nulo declarado da origem está em rating_count —
volume de avaliações (T6), nunca vendas. Os 2 produtos afetados saem da base analítica.


---
## 4. Duplicidade — a inconsistência que define o *n* do projeto (L6)

A granularidade declarada da origem é **uma linha por produto**. Ela não se sustenta: há
`product_id` repetido. Duas medidas distintas importam:

- **linha integralmente duplicada** (todas as 16 colunas iguais) — se houvesse, bastaria `drop_duplicates()`;
- **linha excedente por `product_id`** — o mesmo produto aparece mais de uma vez com **conteúdo diferente**,
  e aí é preciso uma **regra declarada** de qual ocorrência fica.

O caso crítico é o `product_id` cujo `rating_count` **diverge** entre ocorrências: é ele que torna a
regra de desempate obrigatória, porque a escolha muda o número publicado.

In [7]:
dup_integrais = int(df.duplicated().sum())
ids_unicos    = df.product_id.nunique()
excedentes    = len(df) - ids_unicos
ids_repetidos = int((df.product_id.value_counts() > 1).sum())

print(f"base: bruta · n = {br(len(df))}")
print(f"linhas integralmente duplicadas (16 colunas iguais) : {dup_integrais}")
print(f"product_id distintos                                : {br(ids_unicos)}")
print(f"linhas excedentes por product_id                    : {excedentes}")
print(f"product_id que aparecem mais de uma vez             : {ids_repetidos}")
print(f"\nreconciliação: {br(len(df))} − {excedentes} excedentes = {br(ids_unicos)} produtos")

base: bruta · n = 1.465
linhas integralmente duplicadas (16 colunas iguais) : 0
product_id distintos                                : 1.351
linhas excedentes por product_id                    : 114
product_id que aparecem mais de uma vez             : 92

reconciliação: 1.465 − 114 excedentes = 1.351 produtos


In [8]:
# O marcador de moeda da origem entra como escape unicode: a premissa do
# projeto é que o símbolo de rupia não apareça em rótulo, visual ou texto
# publicado. Aqui ele é só alvo de remoção.
MOEDA_ORIGEM = "\u20b9"

# Entre as ocorrências do MESMO product_id, o que diverge?
aux = df.assign(_rc=num(df.rating_count, [","]),
                _pd=num(df.discounted_price, [MOEDA_ORIGEM, ","]),
                _pt=num(df.actual_price, [MOEDA_ORIGEM, ","]))

repetidos = aux[aux.product_id.duplicated(keep=False)]

def divergem(coluna):
    """Quantos product_id repetidos têm mais de um valor distinto nesta coluna."""
    return int((repetidos.groupby("product_id")[coluna].nunique(dropna=False) > 1).sum())

diagnostico = pd.DataFrame({
    "campo": ["rating_count (volume de avaliações)", "rating (nota)", "category",
              "product_name", "discounted_price", "actual_price", "review_content"],
    "product_id com valor divergente": [divergem("_rc"), divergem("rating"), divergem("category"),
                                        divergem("product_name"), divergem("_pd"),
                                        divergem("_pt"), divergem("review_content")],
})
print(f"base: linhas repetidas da bruta · {len(repetidos)} linhas em {ids_repetidos} product_id\n")
diagnostico

base: linhas repetidas da bruta · 206 linhas em 92 product_id



,campo,product_id com valor divergente
0,rating_count (volume de avaliações),31
1,rating (nota),0
2,category,0
3,product_name,0
4,discounted_price,3
5,actual_price,1
6,review_content,18


In [9]:
# Qual o tamanho REAL da divergência de rating_count? (L6 — impacto medido)
por_id = repetidos.groupby("product_id")._rc.agg(["min", "max", "nunique"])
# `por_id["nunique"]` pelo nome, nunca `por_id.nunique` — o atributo resolveria
# para o MÉTODO do DataFrame, não para a coluna homônima.
div    = por_id[por_id["nunique"] > 1]
amplitude  = int((div["max"] - div["min"]).sum())        # máx − mín, somado
maior_rel  = ((div["max"] - div["min"]) / div["max"]).max() * 100
mediana_rel = ((div["max"] - div["min"]) / div["max"]).median() * 100

# Duas medidas DIFERENTES, e confundi-las é o erro fácil aqui:
#   amplitude  = soma de (máx − mín) nos 31 divergentes — o tamanho da hesitação da origem;
#   impacto    = o que muda no número PUBLICADO ao trocar a regra adotada (maior ocorrência)
#                pela alternativa óbvia (primeira ocorrência). É esta que L6 declara.
aux_ordem = aux.assign(_ordem=np.arange(len(aux)))
regra_maior = (aux_ordem.sort_values(["_rc", "_ordem"], ascending=[False, True], na_position="last")
                        .drop_duplicates("product_id", keep="first"))
regra_primeira = aux_ordem.sort_values("_ordem").drop_duplicates("product_id", keep="first")

total_aval = int(regra_maior._rc.sum())
impacto = int(regra_maior._rc.sum() - regra_primeira._rc.sum())

print(f"base: linhas repetidas da bruta · {len(repetidos)} linhas em {ids_repetidos} product_id")
print(f"product_id com rating_count divergente            : {len(div)}")
print(f"maior divergência relativa em um único product_id : {br(maior_rel, 2)}%   (esperado 0,17%)")
print(f"divergência relativa mediana                      : {br(mediana_rel, 2)}%")
print(f"amplitude somada (máx − mín) nos divergentes      : {br(amplitude)} avaliações")
print()
print("IMPACTO DA ESCOLHA DA REGRA — maior ocorrência × primeira ocorrência (L6)")
print(f"  soma de avaliações pela regra adotada (maior)   : {br(total_aval)}")
print(f"  soma pela regra alternativa (primeira)          : {br(int(regra_primeira._rc.sum()))}")
print(f"  diferença                                       : {br(impacto)} avaliações"
      f"  ({br(impacto / total_aval * 100, 4)}%)   (esperado 40 · 0,0002%)")
print("\nLeitura (L6): a divergência é IMATERIAL para qualquer métrica agregada — trocar a")
print("regra move 40 avaliações em 23,8 milhões. Ainda assim o desempate é obrigatório:")
print("sem ele a base não é reproduzível. T6: isto é volume de avaliações, nunca venda.")

base: linhas repetidas da bruta · 206 linhas em 92 product_id
product_id com rating_count divergente            : 31
maior divergência relativa em um único product_id : 0,17%   (esperado 0,17%)
divergência relativa mediana                      : 0,00%
amplitude somada (máx − mín) nos divergentes      : 60 avaliações

IMPACTO DA ESCOLHA DA REGRA — maior ocorrência × primeira ocorrência (L6)
  soma de avaliações pela regra adotada (maior)   : 23.802.463
  soma pela regra alternativa (primeira)          : 23.802.423
  diferença                                       : 40 avaliações  (0,0002%)   (esperado 40 · 0,0002%)

Leitura (L6): a divergência é IMATERIAL para qualquer métrica agregada — trocar a
regra move 40 avaliações em 23,8 milhões. Ainda assim o desempate é obrigatório:
sem ele a base não é reproduzível. T6: isto é volume de avaliações, nunca venda.


In [10]:
# Por que o desempate é obrigatório: na maioria dos repetidos o máximo EMPATA.
def maximo_unico(g):
    m = g.max()
    return (g == m).sum() == 1

tem_max_unico = repetidos.groupby("product_id")._rc.apply(maximo_unico)
print(f"product_id repetidos                        : {ids_repetidos}")
print(f"  com máximo de rating_count ÚNICO          : {int(tem_max_unico.sum())}")
print(f"  com EMPATE no máximo (regra não decide)   : {int((~tem_max_unico).sum())}")
print("\nPor isso a regra completa é: maior rating_count e, no empate,")
print("menor índice de linha na origem (ordenação estável). Ver 02_tratamento.")

product_id repetidos                        : 92
  com máximo de rating_count ÚNICO          : 27
  com EMPATE no máximo (regra não decide)   : 65

Por isso a regra completa é: maior rating_count e, no empate,
menor índice de linha na origem (ordenação estável). Ver 02_tratamento.


---
## 5. Falhas de conversão numérica

Aqui a origem deixa de ser texto. Cada falha é um **produto que não poderá entrar** em alguma análise,
e por isso precisa ser contada antes de ser removida: é a diferença entre "1.348" ser um número
explicado e ser um número herdado.

O que se espera encontrar, pelo dicionário de dados: **1** `rating` inválido (o valor `|`) e
**2** `rating_count` nulos — sem sobreposição entre eles.

In [11]:
# O marcador de moeda da origem entra como escape unicode: a premissa do
# projeto é que o símbolo de rupia não apareça em rótulo, visual ou texto
# publicado. Aqui ele é só alvo de remoção.
MOEDA_ORIGEM = "\u20b9"

conv = pd.DataFrame(index=df.index)
conv["preco_desc"] = num(df.discounted_price, [MOEDA_ORIGEM, ","])
conv["preco_tab"]  = num(df.actual_price,     [MOEDA_ORIGEM, ","])
conv["desconto"]   = num(df.discount_percentage, ["%"])
conv["avaliacoes"] = num(df.rating_count,     [","])
conv["nota"]       = pd.to_numeric(df.rating, errors="coerce")

falhas = pd.DataFrame({
    "coluna de origem": ["discounted_price", "actual_price", "discount_percentage",
                         "rating_count", "rating"],
    "campo tratado": ["preco_desc", "preco_tab", "desconto", "avaliacoes", "nota"],
    "falhas de conversão": [int(conv[c].isna().sum()) for c in
                            ["preco_desc", "preco_tab", "desconto", "avaliacoes", "nota"]],
})
print(f"base: bruta · n = {br(len(df))}\n")
falhas

base: bruta · n = 1.465



,coluna de origem,campo tratado,falhas de conversão
0,discounted_price,preco_desc,0
1,actual_price,preco_tab,0
2,discount_percentage,desconto,0
3,rating_count,avaliacoes,2
4,rating,nota,1


In [12]:
# Que valor, exatamente, impediu a conversão de `rating`?
ruins = df.loc[conv.nota.isna(), ["product_id", "rating"]]
print("registros com `rating` não numérico (conteúdo literal da origem):")
for _, r in ruins.iterrows():
    print(f"  product_id={r.product_id}  rating={r.rating!r}")

sobrepoe = int((conv.nota.isna() & conv.avaliacoes.isna()).sum())
print(f"\nregistros com falha simultânea em rating e rating_count: {sobrepoe}")
print("sem sobreposição → as exclusões se somam: −1 (nota) e −2 (engajamento).")

registros com `rating` não numérico (conteúdo literal da origem):
  product_id=B08L12N5H1  rating='|'

registros com falha simultânea em rating e rating_count: 0
sem sobreposição → as exclusões se somam: −1 (nota) e −2 (engajamento).


---
## 6. Coerência de preço

Teste de sanidade elementar: **preço com desconto nunca pode superar o preço de tabela**. Se
superasse, o campo `desconto` estaria descrevendo outra coisa, e toda a análise comercial ruiria.
A verificação vale a pena justamente porque o resultado esperado é zero — é ela que autoriza usar
o preço sem ressalva.

In [13]:
validos = conv.preco_desc.notna() & conv.preco_tab.notna()
acima   = int((conv.preco_desc[validos] > conv.preco_tab[validos]).sum())
iguais  = int((conv.preco_desc[validos] == conv.preco_tab[validos]).sum())
zeros   = int((conv.preco_tab[validos] == 0).sum())

print(f"base: bruta · n = {br(int(validos.sum()))} linhas com ambos os preços convertidos")
print(f"preço com desconto ACIMA do preço de tabela : {acima}  ← esperado 0")
print(f"preço com desconto IGUAL ao de tabela        : {iguais}  (desconto de 0%)")
print(f"preço de tabela igual a zero                 : {zeros}  (impediria o recálculo)")
print(f"\ncoerência de preço → {'OK' if acima == 0 and zeros == 0 else 'REVISAR'}")
print(MOEDA)

base: bruta · n = 1.465 linhas com ambos os preços convertidos
preço com desconto ACIMA do preço de tabela : 0  ← esperado 0
preço com desconto IGUAL ao de tabela        : 48  (desconto de 0%)
preço de tabela igual a zero                 : 0  (impediria o recálculo)

coerência de preço → OK
valores monetários em rupia indiana, sem conversão e sem símbolo (L5)


---
## 7. Desconto declarado × desconto recalculado

O campo `discount_percentage` é **declarado** pela origem. Recalcular
`(1 − preço com desconto / preço de tabela) × 100` e comparar responde se os três campos monetários
contam a mesma história. Divergência pequena é esperada: a origem publica o desconto **arredondado
para inteiro**, então até meio ponto percentual é efeito de arredondamento, não erro.

O limiar de decisão é: **nenhum caso acima de 1 p.p.** Se houvesse, o campo declarado não poderia
ser usado sem ressalva.

In [14]:
rec = (1 - conv.preco_desc / conv.preco_tab) * 100
diferenca = (rec - conv.desconto).abs()

print(f"base: bruta · n = {br(int(diferenca.notna().sum()))} linhas comparáveis")
print(f"divergência máxima : {br(diferenca.max(), 2)} p.p.")
print(f"divergência média  : {br(diferenca.mean(), 2)} p.p.")
print(f"mediana            : {br(diferenca.median(), 2)} p.p.")
print(f"casos acima de 1 p.p.: {int((diferenca > 1).sum())}  ← esperado 0")
print(f"casos acima de 0,5 p.p.: {int((diferenca > 0.5).sum())}")
print("\nLeitura: a divergência é compatível com arredondamento do percentual declarado")
print("para inteiro. O campo `discount_percentage` pode ser usado como está.")

faixas_div = pd.cut(diferenca.dropna(), [-0.001, 0.1, 0.2, 0.3, 0.4, 0.5],
                    labels=["≤0,1", "0,1–0,2", "0,2–0,3", "0,3–0,4", "0,4–0,5"])
faixas_div.value_counts().sort_index().rename("linhas").to_frame()

base: bruta · n = 1.465 linhas comparáveis
divergência máxima : 0,50 p.p.
divergência média  : 0,21 p.p.
mediana            : 0,18 p.p.
casos acima de 1 p.p.: 0  ← esperado 0
casos acima de 0,5 p.p.: 0

Leitura: a divergência é compatível com arredondamento do percentual declarado
para inteiro. O campo `discount_percentage` pode ser usado como está.


,linhas
"≤0,1",534
"0,1–0,2",226
"0,2–0,3",204
"0,3–0,4",235
"0,4–0,5",266


---
## 8. Da origem para a base analítica — apenas para descrever a distribuição

As seções 2 a 7 descrevem o **bronze**. Distribuição e outliers, porém, só fazem sentido sobre
**uma linha por produto**: com duplicidade, um produto repetido pesa duas vezes no histograma e
desloca quartis. Por isso, e **só a partir daqui**, aplicamos a regra de deduplicação para obter a
base analítica de referência.

Isto **não é o tratamento** — o tratamento, com suas decisões e sua gravação em `silver/` e `gold/`,
é o notebook `02_tratamento`. Aqui a regra é aplicada apenas para que a distribuição seja a correta,
e as cinco bases oficiais ficam reconciliadas na tela.

In [15]:
base = df.join(conv)
base["_ordem"] = np.arange(len(base))     # ordenação estável para o desempate

# Regra declarada: maior rating_count; no empate, menor índice de linha na origem.
produtos = (base.sort_values(["avaliacoes", "_ordem"], ascending=[False, True], na_position="last")
                .drop_duplicates("product_id", keep="first"))

nota_valida  = produtos[produtos.nota.notna()]
engaj_valido = produtos[produtos.avaliacoes.notna()]
an           = produtos[produtos.nota.notna() & produtos.avaliacoes.notna()].copy()

bases = pd.DataFrame({
    "base": ["Bruta", "Produtos", "Nota válida", "Engajamento válido", "Analítica"],
    "n": [len(df), len(produtos), len(nota_valida), len(engaj_valido), len(an)],
    "esperado": [1465, 1351, 1350, 1349, 1348],
    "quando usar": ["qualidade da origem", "contagem de portfólio",
                    "análises que dependem só de rating",
                    "análises que dependem só de rating_count",
                    "PADRÃO — toda análise cruzada"],
})
bases["confere"] = np.where(bases.n == bases.esperado, "OK", "DIVERGENTE")
N = len(an)
bases

,base,n,esperado,quando usar,confere
0,Bruta,1465,1465,qualidade da origem,OK
1,Produtos,1351,1351,contagem de portfólio,OK
2,Nota válida,1350,1350,análises que dependem só de rating,OK
3,Engajamento válido,1349,1349,análises que dependem só de rating_count,OK
4,Analítica,1348,1348,PADRÃO — toda análise cruzada,OK


---
## 9. Outliers pelo critério IQR (cerca de Tukey)

Cerca de Tukey: um valor é **atípico** quando fica fora de `[Q1 − 1,5·IQR ; Q3 + 1,5·IQR]`.

Duas leituras precisam ser separadas, e é por isso que **nada é removido aqui**:

- em **preço** e em **avaliações**, a cauda longa é a **forma natural** da distribuição de um catálogo
  de marketplace — o produto de 139.900 não é erro de digitação, é um produto caro. Remover esses
  pontos apagaria exatamente o fenômeno que a análise de portfólio quer descrever;
- em **nota**, os pontos abaixo da cerca inferior são os produtos de satisfação baixa — o alvo de Q5,
  não ruído.

Outlier estatístico ≠ dado inválido. A EDA mede; quem decide é o escopo analítico.

In [16]:
def cerca(serie):
    """Cerca de Tukey. Devolve limites e contagens fora — sem remover nada."""
    s = serie.dropna()
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iqr = q3 - q1
    inf, sup = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    return dict(q1=q1, q3=q3, iqr=iqr, inf=inf, sup=sup,
                abaixo=int((s < inf).sum()), acima=int((s > sup).sum()), n=len(s))

VARS = {"preco_tab": "Preço de tabela", "preco_desc": "Preço com desconto",
        "desconto": "Desconto (%)", "nota": "Nota", "avaliacoes": "Avaliações (volume)"}

linhas = []
for col, rotulo in VARS.items():
    c = cerca(an[col])
    linhas.append({"variável": rotulo, "n": c["n"], "Q1": c["q1"], "Q3": c["q3"], "IQR": c["iqr"],
                   "cerca inferior": round(c["inf"], 1), "cerca superior": round(c["sup"], 1),
                   "abaixo": c["abaixo"], "acima": c["acima"],
                   "% fora": round((c["abaixo"] + c["acima"]) / c["n"] * 100, 1)})
outliers = pd.DataFrame(linhas)
print(f"base: analítica · n = {br(N)} · critério: cerca de Tukey (1,5 × IQR) · nada foi removido")
print(MOEDA + "\n")
outliers

base: analítica · n = 1.348 · critério: cerca de Tukey (1,5 × IQR) · nada foi removido
valores monetários em rupia indiana, sem conversão e sem símbolo (L5)



,variável,n,Q1,Q3,IQR,cerca inferior,cerca superior,abaixo,acima,% fora
0,Preço de tabela,1348,899.0,4592.5,3693.5,-4641.2,10132.8,0,185,13.7
1,Preço com desconto,1348,349.0,2184.0,1835.0,-2403.5,4936.5,0,209,15.5
2,Desconto (%),1348,31.0,62.0,31.0,-15.5,108.5,0,0,0.0
3,Nota,1348,3.9,4.3,0.4,3.3,4.9,15,2,1.3
4,Avaliações (volume),1348,1107.5,16051.5,14944.0,-21308.5,38467.5,0,130,9.6


---
## 10. Distribuições

A tabela abaixo é a **fonte de verdade das distribuições** do projeto: é contra ela que todo corte
operacional é conferido. Os dois cortes de engajamento saem daqui — **p90 = 36.384** (alto) e
**mediana = 4.740** (baixo) — e os de satisfação, da coluna de nota: **nota < 4,0** e **nota ≥ 4,3**.

Preço e avaliações são **fortemente assimétricos à direita**: média muito acima da mediana. Essa é a
razão de os cortes serem definidos por **quantil**, e não por média — a média, nessas variáveis, não
descreve nenhum produto típico.

In [17]:
dist = an[list(VARS)].describe(percentiles=[.25, .5, .75, .90]).T
dist = dist.rename(index=VARS)[["count", "mean", "std", "min", "25%", "50%", "75%", "90%", "max"]]
dist.columns = ["n", "média", "desvio", "mín", "p25", "mediana", "p75", "p90", "máx"]
print(f"base: analítica · n = {br(N)}")
print(MOEDA + "\n")
dist.round(2)

base: analítica · n = 1.348
valores monetários em rupia indiana, sem conversão e sem símbolo (L5)



,n,média,desvio,mín,p25,mediana,p75,p90,máx
Preço de tabela,1348.0,5700.51,11229.36,39.0,899.0,1795.0,4592.5,15999.0,139900.0
Preço com desconto,1348.0,3310.12,7180.93,39.0,349.0,899.0,2184.0,8992.7,77990.0
Desconto (%),1348.0,46.66,21.61,0.0,31.0,49.0,62.0,75.0,94.0
Nota,1348.0,4.09,0.30,2.0,3.9,4.1,4.3,4.4,5.0
Avaliações (volume),1348.0,17656.88,42158.85,2.0,1107.5,4740.0,16051.5,36384.0,426973.0


In [18]:
# Cortes operacionais que nascem desta distribuição
P90 = an.avaliacoes.quantile(0.90)
MED = an.avaliacoes.median()
baixa = int((an.nota < 4.0).sum())
alta  = int((an.nota >= 4.3).sum())

print(f"base: analítica · n = {br(N)}")
print(f"engajamento alto   ≥ p90 de avaliações : {br(P90)}   (esperado 36.384)")
print(f"engajamento baixo  < mediana           : {br(MED)}    (esperado 4.740)")
print(f"satisfação baixa   nota < 4,0          : {br(baixa)} produtos · {br(baixa / N * 100, 1)}%  (esperado 339 · 25,1%)")
print(f"satisfação alta    nota ≥ 4,3          : {br(alta)} produtos          (esperado 418)")
print("\nT6: `avaliações` é VOLUME DE AVALIAÇÕES — proxy de engajamento, nunca de vendas (L4).")
print("assimetria (Fisher): preço de tabela "
      f"{br(an.preco_tab.skew(), 2)} · avaliações {br(an.avaliacoes.skew(), 2)} · "
      f"nota {br(an.nota.skew(), 2)}")

base: analítica · n = 1.348
engajamento alto   ≥ p90 de avaliações : 36.384   (esperado 36.384)
engajamento baixo  < mediana           : 4.740    (esperado 4.740)
satisfação baixa   nota < 4,0          : 339 produtos · 25,1%  (esperado 339 · 25,1%)
satisfação alta    nota ≥ 4,3          : 418 produtos          (esperado 418)

T6: `avaliações` é VOLUME DE AVALIAÇÕES — proxy de engajamento, nunca de vendas (L4).
assimetria (Fisher): preço de tabela 4,43 · avaliações 5,69 · nota -1,25


### Dimensão do corpus textual (T8)

O bloco de avaliações vem concatenado numa única célula. Antes de qualquer NLP, é preciso declarar
**a unidade** e **o efeito do truncamento** — é isso que T8 exige e o que L1 limita. Duas contagens
muito diferentes convivem: a de *referências* (o que sai do `explode`) e a de *reviews individuais*
(após deduplicar por `review_id`). Publicar a primeira infla o corpus em 14%.

In [19]:
ex = an.assign(_r=an.review_id.str.split(",")).explode("_r")
ex["_r"] = ex._r.str.strip()

referencias = len(ex)
distintos   = ex._r.nunique()
por_produto = an.review_id.str.split(",").apply(len)

print(f"base: analítica · n = {br(N)} produtos")
print(f"referências de avaliação (após explodir)      : {br(referencias)}   (esperado 10.588)")
print(f"reviews individuais (distintos por review_id) : {br(distintos)}    (esperado 9.255)")
print(f"referências repetidas                          : {br(referencias - distintos)}    (esperado 1.333)")
print(f"truncamento — média por produto : {br(por_produto.mean(), 2)}   máximo: {por_produto.max()}")
print("\nT8/L1: a unidade publicável é o REVIEW INDIVIDUAL (deduplicado por review_id),")
print("e o corpus está truncado em 8 avaliações por produto — é a vitrine, não a opinião.")

base: analítica · n = 1.348 produtos
referências de avaliação (após explodir)      : 10.588   (esperado 10.588)
reviews individuais (distintos por review_id) : 9.255    (esperado 9.255)
referências repetidas                          : 1.333    (esperado 1.333)
truncamento — média por produto : 7,85   máximo: 8

T8/L1: a unidade publicável é o REVIEW INDIVIDUAL (deduplicado por review_id),
e o corpus está truncado em 8 avaliações por produto — é a vitrine, não a opinião.


---
## 11. Figuras

Três figuras, todas na paleta v06 aplicada por `paleta.aplicar()` (T9), todas com fonte, base e *n*
no rodapé (T3), todas gravadas em `02_results/images/` com o prefixo `fig_eda_` — que não colide com
`fig01` a `fig13`, geradas por `src/gera_figuras.py`.

### Figura 1 — qualidade da origem
Uma leitura só: **o que está quebrado na origem, e quanto**. As verificações cujo resultado é zero
ficam no gráfico de propósito — um teste que passou é resultado, e omiti-lo daria a impressão de que
não foi feito.

In [20]:
checagens = [
    ("Linhas excedentes por product_id",          excedentes,     True),
    ("product_id repetidos",                      ids_repetidos,  True),
    ("product_id com rating_count divergente",    len(div),       True),
    ("rating_count nulo na origem",               int(conv.avaliacoes.isna().sum()), True),
    ("rating inválido (não numérico)",            int(conv.nota.isna().sum()),       True),
    ("Linhas integralmente duplicadas",           dup_integrais,  False),
    ("Preço com desconto acima do de tabela",     acima,          False),
    ("Divergência de desconto acima de 1 p.p.",   int((diferenca > 1).sum()), False),
]
checar_series(2)   # T9 — duas séries: achado e verificação sem ocorrência

rotulos = [c[0] for c in checagens][::-1]
valores = [c[1] for c in checagens][::-1]
cores   = [(LARANJA if c[2] else TEAL) for c in checagens][::-1]

fig, ax = plt.subplots(figsize=(9.2, 4.4))
barras = ax.barh(rotulos, valores, color=cores, height=0.62)
for barra, valor in zip(barras, valores):
    ax.text(barra.get_width() + max(valores) * 0.012, barra.get_y() + barra.get_height() / 2,
            br(valor), va="center", fontsize=9, color=TINTA, fontweight="bold")
ax.set_xlim(0, max(valores) * 1.16)
ax.set_xlabel("ocorrências na base bruta")
ax.set_title("Qualidade da origem: o que o amazon.csv traz quebrado — e o que não traz")
ax.grid(axis="y", visible=False)

from matplotlib.patches import Patch
ax.legend(handles=[Patch(facecolor=LARANJA, label="achado que exige regra declarada"),
                   Patch(facecolor=TEAL,    label="verificação sem ocorrência")],
          loc="lower right", fontsize=8)

rodape(fig, FONTE + f" · base bruta n = {br(len(df))} · nada foi removido nesta etapa", folga=0.22)
salva(fig, "fig_eda_qualidade_origem.png")

  gravada: 02_results/images/fig_eda_qualidade_origem.png


### Figura 2 — distribuições
Preço e avaliações aparecem em **escala logarítmica**: em escala linear, a cauda longa comprime
todos os produtos num único traço junto ao eixo e o gráfico deixa de informar. A linha vertical marca
a **mediana** — a medida de centro apropriada a distribuições assimétricas.

In [21]:
paineis = [
    ("preco_tab",  "Preço de tabela (rupia indiana)", True,  AZUL),
    ("preco_desc", "Preço com desconto (rupia indiana)", True, TEAL),
    ("desconto",   "Desconto declarado (%)",          False, AMBAR),
    ("nota",       "Nota (1 a 5)",                    False, LARANJA),
    ("avaliacoes", "Avaliações — volume (T6)",        True,  CLARA),
]
checar_series(5)   # T9 — limite de 5 séries respeitado

fig, eixos = plt.subplots(2, 3, figsize=(12.6, 6.4))
eixos = eixos.ravel()

for eixo, (col, titulo, log, cor) in zip(eixos, paineis):
    s = an[col].dropna()
    if log:
        s = s[s > 0]
        bins = np.logspace(np.log10(s.min()), np.log10(s.max()), 34)
    else:
        bins = 30
    eixo.hist(s, bins=bins, color=cor, edgecolor=BASE["card"], linewidth=0.4)
    if log:
        eixo.set_xscale("log")
    eixo.axvline(s.median(), color=ESTRUTURA["tinta"], linestyle="--", linewidth=1.2)
    eixo.text(s.median(), eixo.get_ylim()[1] * 0.94, f" mediana {br(s.median(), 2 if col == 'nota' else 0)}",
              fontsize=8, color=TINTA, ha="left", va="top")
    eixo.set_title(titulo, fontsize=10)
    eixo.set_ylabel("produtos", fontsize=8)

eixos[-1].axis("off")
eixos[-1].text(0.0, 0.62,
               "Escala logarítmica em preço e avaliações.\n"
               "Em escala linear a cauda longa comprime\n"
               "o catálogo inteiro contra o eixo.\n\n"
               f"Assimetria (Fisher):\n"
               f"  preço de tabela  {an.preco_tab.skew():.2f}\n"
               f"  avaliações       {an.avaliacoes.skew():.2f}\n"
               f"  nota             {an.nota.skew():.2f}\n\n"
               "Cortes por quantil, nunca por média.",
               fontsize=8.5, color=SECUNDARIO, va="top", family="monospace")

fig.suptitle("Distribuição das cinco variáveis numéricas — base analítica", fontsize=13, color=TINTA)
fig.tight_layout(rect=[0, 0.045, 1, 0.96])
rodape(fig, FONTE + f" · base analítica n = {br(N)} · {MOEDA}")
salva(fig, "fig_eda_distribuicoes.png")

  gravada: 02_results/images/fig_eda_distribuicoes.png


### Figura 3 — dispersão e cerca de Tukey
Diagrama de caixa por variável, com a **contagem de pontos fora da cerca** anotada. A figura existe
para sustentar uma afirmação do texto: os pontos fora da cerca em preço e em avaliações **não são
erro** — são a cauda do catálogo. **Nenhum deles foi removido.**

In [22]:
fig, eixos = plt.subplots(1, 4, figsize=(12.6, 4.3))
config = [
    ("preco_tab",  "Preço de tabela",     True,  AZUL),
    ("preco_desc", "Preço com desconto",  True,  TEAL),
    ("avaliacoes", "Avaliações (volume)", True,  CLARA),
    ("nota",       "Nota",                False, LARANJA),
]
checar_series(4)   # T9

for eixo, (col, titulo, log, cor) in zip(eixos, config):
    s = an[col].dropna()
    c = cerca(s)
    caixa = eixo.boxplot(s, widths=0.42, patch_artist=True,
                         flierprops=dict(marker="o", markersize=2.4, alpha=0.35,
                                         markerfacecolor=cor, markeredgecolor="none"),
                         medianprops=dict(color=TINTA, linewidth=1.6),
                         whiskerprops=dict(color=SECUNDARIO), capprops=dict(color=SECUNDARIO))
    caixa["boxes"][0].set_facecolor(cor)
    caixa["boxes"][0].set_edgecolor(SECUNDARIO)
    caixa["boxes"][0].set_alpha(0.85)
    if log:
        eixo.set_yscale("log")
    eixo.set_title(titulo, fontsize=10)
    eixo.set_xticks([])
    fora = c["abaixo"] + c["acima"]
    eixo.set_xlabel(f"fora da cerca: {fora}\n({br(fora / c['n'] * 100, 1)}% de {br(c['n'])})",
                    fontsize=8.5, color=SECUNDARIO)

fig.suptitle("Dispersão e cerca de Tukey (1,5 × IQR) — nenhum ponto foi removido",
             fontsize=13, color=TINTA)
fig.tight_layout(rect=[0, 0.06, 1, 0.94])
rodape(fig, FONTE + f" · base analítica n = {br(N)} · critério IQR · {MOEDA}")
salva(fig, "fig_eda_outliers.png")

  gravada: 02_results/images/fig_eda_outliers.png


---
## 12. Validação final — reconciliação das contagens

Última célula do notebook, e a única que pode **falhar em voz alta**. Cada valor medido nesta execução
é comparado ao valor de referência de `method/definitions/base_factual_e_bases_de_calculo.md`. Se
qualquer um divergir, a célula levanta `AssertionError`: um notebook que roda até o fim com números
errados é pior do que um notebook que quebra.

In [23]:
verificacoes = [
    # (descrição,                                       medido,                     esperado)
    ("linhas da base bruta",                            len(df),                    1465),
    ("colunas da base bruta",                           df.shape[1],                16),
    ("linhas integralmente duplicadas",                 dup_integrais,              0),
    ("linhas excedentes por product_id",                excedentes,                 114),
    ("product_id repetidos",                            ids_repetidos,              92),
    ("product_id com rating_count divergente",          len(div),                   31),
    ("impacto da regra (maior × primeira), avaliações",  impacto,                    40),
    ("amplitude somada (máx − mín) nos divergentes",     amplitude,                  60),
    ("rating inválido (falha de conversão)",            int(conv.nota.isna().sum()), 1),
    ("rating_count nulo",                               int(conv.avaliacoes.isna().sum()), 2),
    ("falha de conversão em preços e desconto",         int(conv[["preco_desc", "preco_tab", "desconto"]].isna().sum().sum()), 0),
    ("sobreposição entre as duas falhas",               sobrepoe,                   0),
    ("preço com desconto acima do de tabela",           acima,                      0),
    ("divergência de desconto acima de 1 p.p.",         int((diferenca > 1).sum()), 0),
    ("base Produtos (deduplicada)",                     len(produtos),              1351),
    ("base Nota válida",                                len(nota_valida),           1350),
    ("base Engajamento válido",                         len(engaj_valido),          1349),
    ("base Analítica",                                  N,                          1348),
    ("corte p90 de avaliações",                         int(P90),                   36384),
    ("corte mediana de avaliações",                     int(MED),                   4740),
    ("produtos com nota < 4,0",                         baixa,                      339),
    ("produtos com nota >= 4,3",                        alta,                       418),
    ("referências de avaliação",                        referencias,                10588),
    ("reviews individuais (distintos)",                 distintos,                  9255),
    ("referências repetidas",                           referencias - distintos,    1333),
    ("truncamento máximo por produto",                  int(por_produto.max()),     8),
]

validacao = pd.DataFrame(verificacoes, columns=["verificação", "medido", "esperado"])
validacao["confere"] = np.where(validacao.medido == validacao.esperado, "OK", "DIVERGENTE")

# valores não inteiros conferidos com tolerância
extras = [
    ("divergência máxima de desconto (p.p.)", round(float(diferenca.max()), 2), 0.50, 0.005),
    ("divergência média de desconto (p.p.)",  round(float(diferenca.mean()), 2), 0.21, 0.005),
    ("truncamento médio por produto",         round(float(por_produto.mean()), 2), 7.85, 0.005),
    ("% de produtos com nota < 4,0",          round(baixa / N * 100, 1),        25.1, 0.05),
    ("maior divergência relativa de avaliações (%)", round(float(maior_rel), 2),  0.17, 0.005),
]

print("RECONCILIAÇÃO DAS BASES")
print(f"  {br(len(df))} (bruta) − {excedentes} excedentes = {br(len(produtos))} (produtos)")
print(f"  {br(len(produtos))} − 1 (rating inválido) = {br(len(nota_valida))} (nota válida)")
print(f"  {br(len(produtos))} − 2 (rating_count nulo) = {br(len(engaj_valido))} (engajamento válido)")
print(f"  {br(len(produtos))} − 3 (sem sobreposição) = {br(N)} (ANALÍTICA)\n")

print("CONFERÊNCIA CONTRA A BASE FACTUAL")
for _, r in validacao.iterrows():
    print(f"  [{r.confere:^10}] {r['verificação']:<44} medido {r.medido:>7} · esperado {r.esperado:>7}")
for desc, medido, esperado, tol in extras:
    ok = "OK" if abs(medido - esperado) <= tol else "DIVERGENTE"
    print(f"  [{ok:^10}] {desc:<44} medido {medido:>7} · esperado {esperado:>7}")

figuras = sorted(p.name for p in IMG.glob("fig_eda_*.png"))
print(f"\nFIGURAS GERADAS ({len(figuras)}) em 02_results/images/")
for f in figuras:
    print(f"  {f}")

divergentes = validacao[validacao.confere != "OK"]
falhas_extras = [d for d, m, e, t in extras if abs(m - e) > t]
assert divergentes.empty, f"contagens divergentes: {divergentes.to_dict('records')}"
assert not falhas_extras, f"valores contínuos divergentes: {falhas_extras}"
assert len(figuras) == 3, f"esperadas 3 figuras fig_eda_, encontradas {len(figuras)}"

print(f"\nTODAS AS {len(validacao) + len(extras)} VERIFICAÇÕES CONFEREM.")
print("EDA concluída — nenhum dado foi alterado. Próximo passo: 02_tratamento.ipynb")

RECONCILIAÇÃO DAS BASES
  1.465 (bruta) − 114 excedentes = 1.351 (produtos)
  1.351 − 1 (rating inválido) = 1.350 (nota válida)
  1.351 − 2 (rating_count nulo) = 1.349 (engajamento válido)
  1.351 − 3 (sem sobreposição) = 1.348 (ANALÍTICA)

CONFERÊNCIA CONTRA A BASE FACTUAL
  [    OK    ] linhas da base bruta                         medido    1465 · esperado    1465
  [    OK    ] colunas da base bruta                        medido      16 · esperado      16
  [    OK    ] linhas integralmente duplicadas              medido       0 · esperado       0
  [    OK    ] linhas excedentes por product_id             medido     114 · esperado     114
  [    OK    ] product_id repetidos                         medido      92 · esperado      92
  [    OK    ] product_id com rating_count divergente       medido      31 · esperado      31
  [    OK    ] impacto da regra (maior × primeira), avaliações medido      40 · esperado      40
  [    OK    ] amplitude somada (máx − mín) nos divergentes medi

---
## Conclusão da EDA

**O que a origem é.** 1.465 linhas, 16 colunas, tudo texto. Nenhuma linha integralmente duplicada, mas
**114 linhas excedentes** em **92 `product_id`** — a granularidade declarada ("uma linha por produto")
não se sustenta sem uma regra explícita de deduplicação (L6).

**O que está quebrado, e o quanto.** Um `rating` não numérico e dois `rating_count` nulos: três
produtos, sem sobreposição, que não entram na base analítica. Nenhuma falha de conversão em preço ou
desconto. Zero casos de preço com desconto acima do preço de tabela. Divergência entre desconto
declarado e recalculado de no máximo **0,50 p.p.** — arredondamento, não inconsistência.

**O que é forma, não erro.** Preço e volume de avaliações são fortemente assimétricos à direita. Os
pontos fora da cerca de Tukey são a **cauda do catálogo**, e em nota são os produtos de satisfação
baixa — o alvo da análise, não ruído. **Nada foi removido nesta etapa.**

**O que fica decidido pela distribuição.** Os cortes operacionais são de **quantil**, não de média:
engajamento alto em **36.384** (p90) e baixo em **4.740** (mediana); satisfação baixa em **nota < 4,0**
(339 produtos, 25,1%) e alta em **nota ≥ 4,3** (418).

**O que continua vedado.** A base não tem venda, receita nem data (L4): `rating_count` é **volume de
avaliações** (T6) e não existe análise temporal. Associações medidas adiante indicam **associação**,
nunca causa (T4), sempre com magnitude ao lado da significância (T5). E o corpus textual está
**truncado em 8 avaliações por produto** (L1): descreve a vitrine, não a opinião.

> Próximo notebook: **`02_tratamento.ipynb`** — aplica a regra de deduplicação com desempate declarado,
> converte os tipos e grava `00_data/silver/produtos.csv` e `00_data/gold/base_analitica.csv`.